# Introduction to text embeddings on S&P 500 news
### Course Project — Part 1, Section A

**Pipeline:** S&P 500 tickers → `yfinance` news → structured DataFrame → `all-MiniLM-L6-v2`
embeddings → K-Means clustering → silhouette-based choice of *k* → PCA visualisation →
interpretation of the clusters.

# 📌 Objectives

By the end of this notebook, students will be able to:

1. **Retrieve Financial News:**
   - Use the `yfinance` library to gather news headlines for all companies in the S&P 500 index.

2. **Clean and Structure Financial Text Data:**
   - Extract and organize relevant metadata (e.g., ticker, title, summary, publication date, URL) into a structured pandas DataFrame.

3. **Generate Text Embeddings:**
   - Apply a pre-trained sentence transformer model (`all-MiniLM-L6-v2`) to convert news headlines and summaries into numerical embeddings.

4. **Apply Clustering Techniques:**
   - Use K-Means clustering to identify groups of similar news articles based on semantic content.

5. **Determine Optimal Number of Clusters:**
   - Evaluate clustering quality using silhouette scores to find the best number of clusters.

6. **Visualize High-Dimensional Embeddings:**
   - Reduce the embedding space using PCA and visualize clusters in two dimensions.

7. **Interpret Cluster Themes:**
   - Analyze representative news headlines of every cluster and give each group a business-readable label.

## Install and Import important librairies

In [ ]:
%pip install --quiet pandas numpy yfinance lxml matplotlib scikit-learn
%pip install --quiet -U sentence-transformers

In [ ]:
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import Markdown, display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (calinski_harabasz_score, davies_bouldin_score,
                             silhouette_samples, silhouette_score)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)

## Get the list of stocks in the S&P 500 

In [ ]:
# Read and print the stock tickers that make up S&P500
df_tickers = pd.read_html(
    'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]

display(df_tickers.head())

In [ ]:
ticker_list = df_tickers['Symbol'].tolist()

In [ ]:
# Yahoo Finance uses '-' instead of '.' for share classes (BRK.B -> BRK-B)
yahoo_symbols = [t.strip().replace('.', '-') for t in ticker_list]

# Company / sector metadata: used later to check whether the clusters are just sector groupings
meta = (df_tickers.assign(YAHOO_SYMBOL=yahoo_symbols)
                  .set_index('YAHOO_SYMBOL')[['Security', 'GICS Sector']]
                  .rename(columns={'Security': 'COMPANY', 'GICS Sector': 'SECTOR'}))

print(f"S&P 500 constituents: {len(yahoo_symbols)}")
display(meta.head())

## Get the news of all 500 stocks in the S&P 500 Index
Use the yfinance library to retrieve the news of all 500 stocks in the index.
https://ranaroussi.github.io/yfinance/reference/yfinance.stock.html

### Get the news in a dictionary

The `.news` endpoint is queried once per ticker, so this loop makes ~500 HTTP calls. Three
precautions keep it robust:

* every call is wrapped in a `try/except` — a single failing ticker must not kill the whole loop;
* a short pause between calls avoids being throttled by Yahoo;
* newer `yfinance` releases expose `get_news(count=…)` while older ones only expose the `.news`
  property, so we support both.

Expect a non-trivial number of tickers to return an empty list: not every S&P 500 company has news
attached on a given day.

In [ ]:
def fetch_news(symbol, count=10, pause=0.15):
    """Return the raw list of news items for one ticker (empty list on failure)."""
    try:
        ticker = yf.Ticker(symbol)
        try:
            items = ticker.get_news(count=count)          # yfinance >= 0.2.5x
        except TypeError:
            items = ticker.news                           # older releases
        return items or []
    except Exception as exc:
        print(f"  [warn] {symbol}: {type(exc).__name__} - {exc}")
        return []
    finally:
        time.sleep(pause)


# Dictionary: key = ticker symbol, value = raw news payload returned by yfinance
news_dict = {}
t0 = time.time()
for i, symbol in enumerate(yahoo_symbols, start=1):
    news_dict[symbol] = fetch_news(symbol)
    if i % 50 == 0:
        collected = sum(len(v) for v in news_dict.values())
        print(f"  {i:>3}/{len(yahoo_symbols)} tickers | {collected} articles | "
              f"{time.time() - t0:.0f}s elapsed")

n_articles = sum(len(v) for v in news_dict.values())
n_empty = sum(1 for v in news_dict.values() if not v)
print(f"\nTickers queried      : {len(news_dict)}")
print(f"Tickers with no news : {n_empty}")
print(f"Raw articles collected: {n_articles}")

In [ ]:
# What does one raw item actually look like? (useful to know which keys to extract)
sample_symbol = next(s for s, v in news_dict.items() if v)
sample_item = news_dict[sample_symbol][0]
print("Sample ticker:", sample_symbol)
print("Top-level keys:", list(sample_item.keys()))
if 'content' in sample_item:
    print("content keys :", list(sample_item['content'].keys()))
sample_item

### Structure the news into a pandas dataframe

Your final dataframe should have the following columns:
- TICKER
- TITLE (of the news)
- SUMMARY (of the news)
- PUBLICATION_DATE (of the news)
- URL (of the news)

Note: all of those fields are provided in the yfinance news component. Refer to the library documentation.

**Two payload schemas.** Recent `yfinance` versions nest everything under `content`
(`content.title`, `content.summary`, `content.pubDate`, `content.canonicalUrl.url`,
`content.provider.displayName`), while older versions return a flat dictionary
(`title`, `publisher`, `link`, `providerPublishTime` as a UNIX timestamp). The parser below handles
both so the notebook does not break when the library is upgraded.

In [ ]:
def _first(dct, *keys):
    """Return the first non-empty value among `keys`."""
    for k in keys:
        value = dct.get(k)
        if value:
            return value
    return None


def parse_article(symbol, item):
    """Flatten one raw yfinance news item into a record with the required columns."""
    content = item.get('content', item)                   # new schema nests under 'content'

    title = _first(content, 'title', 'headline')
    summary = _first(content, 'summary', 'description', 'shortDescription')

    # URL: new schema -> canonicalUrl/clickThroughUrl dicts; old schema -> 'link'
    url = None
    for key in ('canonicalUrl', 'clickThroughUrl'):
        node = content.get(key)
        if isinstance(node, dict) and node.get('url'):
            url = node['url']
            break
    url = url or _first(content, 'link', 'url')

    # Provider: new schema -> provider.displayName; old schema -> 'publisher'
    provider = content.get('provider')
    provider = provider.get('displayName') if isinstance(provider, dict) else provider
    provider = provider or _first(content, 'publisher', 'providerDisplayName')

    published = _first(content, 'pubDate', 'displayTime', 'providerPublishTime')

    return {
        'TICKER': symbol,
        'TITLE': title,
        'SUMMARY': summary,
        'PUBLICATION_DATE': published,
        'URL': url,
        'PROVIDER': provider,
        'ARTICLE_ID': item.get('id') or _first(content, 'id', 'uuid'),
    }


records = [parse_article(symbol, item)
           for symbol, items in news_dict.items()
           for item in items]

df_news_raw = pd.DataFrame(records)
print(f"Raw news DataFrame: {df_news_raw.shape[0]} rows x {df_news_raw.shape[1]} columns")
display(df_news_raw.head())

In [ ]:
# --- Cleaning ---------------------------------------------------------------------------------
df_news = df_news_raw.copy()
before = len(df_news)

# 1. Publication date -> timezone-aware datetime (epoch seconds in the legacy schema)
def to_datetime(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, (int, float)) or (isinstance(value, str) and str(value).isdigit()):
        return pd.to_datetime(float(value), unit='s', utc=True)
    return pd.to_datetime(value, utc=True, errors='coerce')

df_news['PUBLICATION_DATE'] = df_news['PUBLICATION_DATE'].apply(to_datetime)

# 2. Normalise the text fields
for col in ['TITLE', 'SUMMARY', 'PROVIDER']:
    df_news[col] = (df_news[col].astype('string')
                                .str.replace(r'\s+', ' ', regex=True)
                                .str.strip())

# 3. A row without a title is unusable (the title is what we embed)
df_news = df_news[df_news['TITLE'].notna() & (df_news['TITLE'].str.len() >= 10)]

# 4. Drop duplicates: the same article is often attached to several tickers
df_news = df_news.drop_duplicates(subset=['TICKER', 'TITLE'])

# 5. Missing summaries: keep the row, flag it, and fall back to the title
df_news['HAS_SUMMARY'] = df_news['SUMMARY'].notna()
df_news['SUMMARY'] = df_news['SUMMARY'].fillna(df_news['TITLE'])

df_news = (df_news.sort_values(['TICKER', 'PUBLICATION_DATE'], ascending=[True, False])
                  .reset_index(drop=True))

print(f"Rows before cleaning : {before}")
print(f"Rows after cleaning  : {len(df_news)}")
print(f"Unique tickers       : {df_news['TICKER'].nunique()}")
print(f"Articles per ticker  : mean {len(df_news) / df_news['TICKER'].nunique():.1f}")
print(f"Rows without a real summary: {(~df_news['HAS_SUMMARY']).sum()}")

In [ ]:
df_news = df_news[['TICKER', 'TITLE', 'SUMMARY', 'PUBLICATION_DATE', 'URL', 'PROVIDER',
                   'ARTICLE_ID', 'HAS_SUMMARY']]
display(df_news.head(10))
df_news.info()

In [ ]:
# Quick descriptive checks on the corpus we are about to embed
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df_news['TITLE'].str.split().str.len().plot(kind='hist', bins=30, ax=axes[0],
                                            color='#4c72b0', edgecolor='white')
axes[0].set_title('Length of the news titles')
axes[0].set_xlabel('Number of words')

top_providers = df_news['PROVIDER'].value_counts().head(10)
axes[1].barh(top_providers.index[::-1].astype(str), top_providers.values[::-1], color='#55a868')
axes[1].set_title('Top 10 news providers')
axes[1].set_xlabel('Number of articles')

plt.tight_layout()
plt.show()

if df_news['PUBLICATION_DATE'].notna().any():
    print("Publication dates range from "
          f"{df_news['PUBLICATION_DATE'].min()} to {df_news['PUBLICATION_DATE'].max()}")

## Exploring text embeddings

- Use the open-source model: 'sentence-transformers/all-MiniLM-L6-v2' to create embeddings on the news title and summary
- Add a column to your news dataframe called EMBEDDED_TEXT using ONLY the TITLE of the news
- Add a column to your news dataframe called EMBEDDINGS, which contains the embedding of EMBEDDED_TEXT

`all-MiniLM-L6-v2` is a 6-layer distilled transformer that maps a sentence to a **384-dimensional**
vector, trained so that semantically similar sentences end up close to each other under cosine
similarity. It truncates inputs at 256 word-pieces — irrelevant here, since news titles are ~10–20
words.

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(model)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

In [ ]:
# EMBEDDED_TEXT = the news TITLE only (as required)
df_news['EMBEDDED_TEXT'] = df_news['TITLE'].astype(str).str.strip()

embeddings = model.encode(df_news['EMBEDDED_TEXT'].tolist(),
                          batch_size=64,
                          show_progress_bar=True,
                          convert_to_numpy=True)

df_news['EMBEDDINGS'] = list(embeddings)

print(f"Embeddings matrix: {embeddings.shape}  (rows = articles, columns = dimensions)")
display(df_news[['TICKER', 'TITLE', 'EMBEDDED_TEXT', 'EMBEDDINGS']].head())

In [ ]:
# Sanity check: does the embedding space actually capture meaning?
# The two most similar titles in a random sample should be semantically related.
from sklearn.metrics.pairwise import cosine_similarity

sample = df_news.sample(min(200, len(df_news)), random_state=RANDOM_STATE).reset_index(drop=True)
sim = cosine_similarity(np.vstack(sample['EMBEDDINGS'].to_numpy()))
np.fill_diagonal(sim, -1)
i, j = np.unravel_index(sim.argmax(), sim.shape)

print(f"Highest cosine similarity in the sample: {sim[i, j]:.3f}")
print(f"  A ({sample.loc[i, 'TICKER']}): {sample.loc[i, 'TITLE']}")
print(f"  B ({sample.loc[j, 'TICKER']}): {sample.loc[j, 'TITLE']}")

## Using K-means clustering on news embeddings
to simplify, keep only one news for each company (ticker), you should have 500 rows in your news dataframe

> **Why the result may be slightly under 500 rows.** One row per ticker is exactly what the cell below
> produces, but `.news` returns an empty list for some constituents on any given day (recent IPOs,
> quiet small caps, transient API failures). The number of rows is therefore
> *the number of tickers that actually had at least one usable article* — reported explicitly below,
> together with the coverage ratio, rather than silently padded to 500.

**One article per ticker is not the same as one article per *story*.** Wire services attach a single
piece to every company it mentions, so a headline such as *"Big Tech's AI shakeout just changed
everything"* can be the most recent article of six different mega-caps at once. Naively keeping the
newest article per ticker would then insert the *same* text six times into the clustering matrix — and
because K-Means centroids are means, that story would pull a centroid with six times the weight it
deserves, manufacturing a cluster out of a duplication artefact.

The selection below therefore walks each ticker in turn and takes its most recent article **whose
title has not already been used**, falling back to the newest one only if all of its articles are
duplicates. Every ticker keeps a row, and no story is counted twice.

In [ ]:
# One article per ticker, avoiding the SAME syndicated story being selected for several tickers
seen_titles, picks = set(), []
ordered = df_news.sort_values(['TICKER', 'PUBLICATION_DATE'], ascending=[True, False])

for ticker, group in ordered.groupby('TICKER', sort=True):
    fresh = group[~group['TITLE'].str.lower().isin(seen_titles)]
    chosen = (fresh if not fresh.empty else group).iloc[0]      # fall back if all are duplicates
    seen_titles.add(str(chosen['TITLE']).lower())
    picks.append(chosen)

df_news_unique = pd.DataFrame(picks).reset_index(drop=True)

n_dupes_avoided = df_news_unique['TICKER'].nunique() - df_news_unique['TITLE'].str.lower().nunique()
print(f"Distinct stories among the selected articles: "
      f"{df_news_unique['TITLE'].str.lower().nunique()} / {len(df_news_unique)} "
      f"({n_dupes_avoided} tickers had only duplicated stories available)")

In [ ]:
df_news_unique['COMPANY'] = df_news_unique['TICKER'].map(meta['COMPANY'])
df_news_unique['SECTOR'] = df_news_unique['TICKER'].map(meta['SECTOR'])

X = np.vstack(df_news_unique['EMBEDDINGS'].to_numpy())

print(f"df_news_unique: {df_news_unique.shape[0]} rows (one news per ticker)")
print(f"Coverage: {df_news_unique.shape[0]}/{len(yahoo_symbols)} constituents "
      f"({df_news_unique.shape[0] / len(yahoo_symbols):.1%}) returned at least one usable article")
print(f"Embedding matrix X: {X.shape}")
display(df_news_unique[['TICKER', 'COMPANY', 'SECTOR', 'TITLE', 'PUBLICATION_DATE']].head(10))

### Identify the number of clusters using the silhouette score

- Using a for loop, do the clustering with different k values (number of clusters), test 1 to 6 clusters
- Compute the silhouette score for every k value
- Plot the silhouette score for different k values

#### Try different values of k and compute silhouette scores

> **Note on k = 1.** The silhouette score of an observation compares its average distance to its own
> cluster with its average distance to the *nearest other* cluster. With a single cluster there is no
> "other cluster", so the score is mathematically undefined (`scikit-learn` raises a `ValueError` for
> `n_labels = 1`). The loop therefore runs over **k = 2 … 6**, which is also the range required by the
> assignment brief.

In [ ]:
k_values = range(2, 7)
rows = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X)
    rows.append({
        'K': k,
        'SILHOUETTE': silhouette_score(X, labels),          # higher is better
        'DAVIES_BOULDIN': davies_bouldin_score(X, labels),  # LOWER is better
        'CALINSKI_HARABASZ': calinski_harabasz_score(X, labels),   # higher is better
        'INERTIA': km.inertia_,
        'SMALLEST_CLUSTER': int(np.bincount(labels).min()),
        'LARGEST_CLUSTER': int(np.bincount(labels).max()),
    })
    r = rows[-1]
    print(f"k = {k} | silhouette = {r['SILHOUETTE']:.4f} | "
          f"davies-bouldin = {r['DAVIES_BOULDIN']:.4f} | "
          f"calinski-harabasz = {r['CALINSKI_HARABASZ']:.2f} | "
          f"inertia = {km.inertia_:,.0f} | sizes = {np.bincount(labels).tolist()}")

df_silhouette = pd.DataFrame(rows)
display(df_silhouette.style.format({
    'SILHOUETTE': '{:.4f}', 'DAVIES_BOULDIN': '{:.4f}',
    'CALINSKI_HARABASZ': '{:.2f}', 'INERTIA': '{:,.0f}'}))

> **Read the three metrics together, and do not assume they agree.** Silhouette (higher = better),
> Davies-Bouldin (**lower** = better) and Calinski-Harabasz (higher = better) measure separation in
> different ways, and on short-text embeddings they frequently point to *different* values of k. When
> they disagree, that disagreement is itself the finding: it means no k is clearly right and the
> structure is weak. The cell below states explicitly which k each metric prefers instead of assuming
> one answer.

In [ ]:
verdicts = {
    'silhouette (max)': int(df_silhouette.loc[df_silhouette['SILHOUETTE'].idxmax(), 'K']),
    'davies-bouldin (min)': int(df_silhouette.loc[df_silhouette['DAVIES_BOULDIN'].idxmin(), 'K']),
    'calinski-harabasz (max)': int(df_silhouette.loc[df_silhouette['CALINSKI_HARABASZ'].idxmax(), 'K']),
}
for metric, k in verdicts.items():
    print(f"  {metric:<26} -> k = {k}")

if len(set(verdicts.values())) == 1:
    print(f"\nAll three metrics agree on k = {list(verdicts.values())[0]}.")
else:
    print("\nThe metrics DISAGREE. No value of k is clearly optimal, which is evidence that the "
          "corpus has weak cluster structure rather than evidence for any particular k.")

#### Plot silhouette scores

In [ ]:
best_k = int(df_silhouette.loc[df_silhouette['SILHOUETTE'].idxmax(), 'K'])
best_score = df_silhouette['SILHOUETTE'].max()

fig, ax1 = plt.subplots(figsize=(11, 5.5))
ax1.plot(df_silhouette['K'], df_silhouette['SILHOUETTE'], 'o-', color='#4c72b0',
         lw=2, ms=9, label='Silhouette score')
ax1.scatter([best_k], [best_score], s=280, facecolor='none', edgecolor='#d62728',
            lw=2.5, zorder=5, label=f'Best k = {best_k}')
# Put the callout on the inner side of the curve so it never runs off the axes
dx = -110 if best_k >= max(k_values) else 14
ax1.annotate(f'best k = {best_k}\nscore = {best_score:.4f}',
             xy=(best_k, best_score), xytext=(dx, -38), textcoords='offset points',
             fontsize=10, color='#d62728', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#d62728'))

for _, r in df_silhouette.iterrows():
    ax1.annotate(f"{r['SILHOUETTE']:.3f}", (r['K'], r['SILHOUETTE']),
                 textcoords='offset points', xytext=(0, 11), ha='center', fontsize=9)

ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Silhouette score (higher = better separated)', color='#4c72b0')
ax1.set_xticks(list(k_values))
ax1.set_ylim(0, max(0.15, best_score * 1.5))

# The elbow curve on a secondary axis, as a cross-check on the silhouette
ax2 = ax1.twinx()
ax2.plot(df_silhouette['K'], df_silhouette['INERTIA'], 's--', color='#999999',
         lw=1.3, ms=6, label='Inertia (elbow method)')
ax2.set_ylabel('K-Means inertia (within-cluster sum of squares)', color='#777777')
ax2.grid(False)

lines = ax1.get_lines()[:1] + [ax1.collections[0]] + ax2.get_lines()
ax1.legend(lines, ['Silhouette score', f'Best k = {best_k}', 'Inertia (elbow)'], loc='best')
ax1.set_title('Choosing the number of clusters: silhouette score and elbow curve',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

#### Identify the Best k

In [ ]:
display(Markdown(
    f"**Best k according to the silhouette score: `k = {best_k}` "
    f"(score = {best_score:.4f})**\n\n"
    f"Ranking of the tested values: "
    + ", ".join(f"k={int(r.K)} ({r.SILHOUETTE:.4f})"
                for r in df_silhouette.sort_values('SILHOUETTE', ascending=False).itertuples())
))

print("\nInterpretation guide for silhouette values:")
print("   > 0.50  strong, well-separated structure")
print("  0.25-0.50 reasonable structure")
print("   < 0.25  weak structure — clusters overlap heavily (usual for short-text embeddings)")

#### Cluster the embeddings using 3 clusters (k=3)

In [ ]:
K_FINAL = 3
kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
labels_k3 = kmeans.fit_predict(X)

# Add the cluster label to the news dataframe (required deliverable)
df_news_unique['CLUSTER'] = labels_k3

# Keep the best-k partition alongside, to compare the two solutions
kmeans_best = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df_news_unique['CLUSTER_BEST_K'] = kmeans_best.fit_predict(X)

print(f"K-Means with k = {K_FINAL} | inertia = {kmeans.inertia_:,.0f} | "
      f"silhouette = {silhouette_score(X, labels_k3):.4f}")
print("\nCluster sizes:")
print(df_news_unique['CLUSTER'].value_counts().sort_index().to_string())
display(df_news_unique[['TICKER', 'COMPANY', 'TITLE', 'CLUSTER', 'CLUSTER_BEST_K']].head(10))

In [ ]:
# Per-observation silhouette: which cluster is compact, and how many points are misassigned?
sil_values = silhouette_samples(X, labels_k3)
df_news_unique['SILHOUETTE'] = sil_values

quality = (df_news_unique.groupby('CLUSTER')['SILHOUETTE']
           .agg(SIZE='size', MEAN_SILHOUETTE='mean', MIN='min', MAX='max'))
quality['NEGATIVE_SILHOUETTE'] = (df_news_unique.assign(neg=sil_values < 0)
                                  .groupby('CLUSTER')['neg'].sum())
display(quality.style.format({'MEAN_SILHOUETTE': '{:.3f}', 'MIN': '{:.3f}', 'MAX': '{:.3f}'}))
print(f"\nArticles with a negative silhouette (closer to another cluster): "
      f"{(sil_values < 0).sum()} / {len(sil_values)} ({(sil_values < 0).mean():.1%})")

#### Optional: Yellowbrick diagnostics (Intercluster Distance Map + Silhouette Visualizer)

Two visual cross-checks on the partition. The **Intercluster Distance Map** projects the centroids and
scales each circle by cluster membership: heavily overlapping circles mean the clusters are not
separated in the embedding space. The **Silhouette Visualizer** draws the silhouette of every single
article, so the share of the corpus falling below zero — the articles that are closer to another
cluster than to their own — is visible at a glance instead of hidden in an average.

The cell degrades gracefully: if `yellowbrick` cannot be installed (it pins older scikit-learn
versions and occasionally conflicts), the notebook keeps running.

In [ ]:
%pip install --quiet yellowbrick

In [ ]:
try:
    from yellowbrick.cluster import InterclusterDistance, SilhouetteVisualizer

    km_vis = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)

    fig, ax = plt.subplots(figsize=(10, 7))
    viz = InterclusterDistance(km_vis, ax=ax, random_state=RANDOM_STATE)
    viz.fit(X)
    viz.show()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 7))
    viz = SilhouetteVisualizer(KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10),
                               colors='yellowbrick', ax=ax)
    viz.fit(X)
    viz.show()
    plt.show()
except Exception as exc:
    print(f"Yellowbrick diagnostics skipped ({type(exc).__name__}: {exc}).")
    print("The silhouette table, the silhouette plot and the per-article silhouette column above "
          "already cover the same diagnostics.")

### Visualize the 2 first PCA Components of your embeddings

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X)
df_news_unique['PCA_1'], df_news_unique['PCA_2'] = coords[:, 0], coords[:, 1]

explained = pca.explained_variance_ratio_
print(f"Variance explained by PC1: {explained[0]:.2%}")
print(f"Variance explained by PC2: {explained[1]:.2%}")
print(f"Total variance captured by the 2D plot: {explained.sum():.2%}")

In [ ]:
PALETTE = ['#4c72b0', '#dd8452', '#55a868', '#c44e52', '#8172b3', '#937860']

fig, ax = plt.subplots(figsize=(12, 8))

for cluster in sorted(df_news_unique['CLUSTER'].unique()):
    sub = df_news_unique[df_news_unique['CLUSTER'] == cluster]
    ax.scatter(sub['PCA_1'], sub['PCA_2'], s=45, alpha=0.75,
               color=PALETTE[cluster % len(PALETTE)], edgecolor='white', linewidth=0.5,
               label=f'Cluster {cluster}  (n = {len(sub)})')

# Centroids projected into the same 2D space
centroids_2d = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroids_2d[:, 0], centroids_2d[:, 1], marker='X', s=420,
           c='black', edgecolor='white', linewidth=1.5, zorder=6, label='Cluster centroid')
for cluster, (x, y) in enumerate(centroids_2d):
    ax.annotate(f'C{cluster}', (x, y), fontsize=13, fontweight='bold', color='black',
                xytext=(12, 10), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.85))

ax.set_xlabel(f'PC 1 — {explained[0]:.1%} of the variance')
ax.set_ylabel(f'PC 2 — {explained[1]:.1%} of the variance')
ax.set_title(f'S&P 500 news headlines in embedding space\n'
             f'K-Means (k = {K_FINAL}) on all-MiniLM-L6-v2 embeddings, projected with PCA '
             f'({explained.sum():.1%} of total variance)',
             fontsize=12, fontweight='bold')
ax.legend(loc='best', framealpha=0.92)
ax.axhline(0, color='grey', lw=0.6)
ax.axvline(0, color='grey', lw=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Same projection, annotated with a few real headlines so the axes become interpretable
fig, ax = plt.subplots(figsize=(13, 9))

for cluster in sorted(df_news_unique['CLUSTER'].unique()):
    sub = df_news_unique[df_news_unique['CLUSTER'] == cluster]
    ax.scatter(sub['PCA_1'], sub['PCA_2'], s=38, alpha=0.45,
               color=PALETTE[cluster % len(PALETTE)], label=f'Cluster {cluster}')

    # Annotate the 3 headlines closest to each centroid (the most "typical" ones)
    centre = np.array([sub['PCA_1'].mean(), sub['PCA_2'].mean()])
    dist = np.linalg.norm(sub[['PCA_1', 'PCA_2']].to_numpy() - centre, axis=1)
    for idx in sub.index[np.argsort(dist)[:3]]:
        row = df_news_unique.loc[idx]
        text = (row['TITLE'][:52] + '…') if len(row['TITLE']) > 52 else row['TITLE']
        ax.annotate(f"[{row['TICKER']}] {text}", (row['PCA_1'], row['PCA_2']),
                    fontsize=7.5, alpha=0.95,
                    color=PALETTE[cluster % len(PALETTE)],
                    xytext=(6, 5), textcoords='offset points')

ax.set_xlabel(f'PC 1 — {explained[0]:.1%} of the variance')
ax.set_ylabel(f'PC 2 — {explained[1]:.1%} of the variance')
ax.set_title('Representative headlines near each cluster centroid', fontsize=12, fontweight='bold')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

#### Analyze the content of each cluster
- Add the kmeans cluster label to your news dataframe
- Print the content of each cluster and analyze it

In [ ]:
def top_terms_per_cluster(df, text_col='TITLE', label_col='CLUSTER', n_terms=12):
    """Terms that are most characteristic of each cluster, via TF-IDF centroids."""
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2),
                          min_df=2, max_features=5000)
    tfidf = vec.fit_transform(df[text_col].astype(str))
    vocab = np.array(vec.get_feature_names_out())

    out = {}
    for cluster in sorted(df[label_col].unique()):
        mask = (df[label_col] == cluster).to_numpy()
        centroid = np.asarray(tfidf[mask].mean(axis=0)).ravel()
        out[cluster] = vocab[centroid.argsort()[::-1][:n_terms]].tolist()
    return out


cluster_terms = top_terms_per_cluster(df_news_unique)
for cluster, terms in cluster_terms.items():
    print(f"\nCluster {cluster} ({(df_news_unique['CLUSTER'] == cluster).sum()} articles)")
    print("  characteristic terms:", ", ".join(terms))

In [ ]:
# Print a readable sample of each cluster: the most typical headlines (highest silhouette)
for cluster in sorted(df_news_unique['CLUSTER'].unique()):
    sub = (df_news_unique[df_news_unique['CLUSTER'] == cluster]
           .sort_values('SILHOUETTE', ascending=False))
    display(Markdown(
        f"### Cluster {cluster} — {len(sub)} articles "
        f"(mean silhouette {sub['SILHOUETTE'].mean():.3f})\n"
        f"**Characteristic terms:** {', '.join(cluster_terms[cluster])}"
    ))
    print("MOST REPRESENTATIVE HEADLINES (closest to the cluster core)")
    for _, row in sub.head(10).iterrows():
        print(f"  [{row['TICKER']:<6}] {row['TITLE']}")
    print("\nLEAST REPRESENTATIVE HEADLINES (borderline cases)")
    for _, row in sub.tail(3).iterrows():
        print(f"  [{row['TICKER']:<6}] {row['TITLE']}  (silhouette {row['SILHOUETTE']:.3f})")
    print()

In [ ]:
# Are the clusters just a re-labelling of GICS sectors? A cross-tab answers directly.
sector_mix = pd.crosstab(df_news_unique['SECTOR'], df_news_unique['CLUSTER'], normalize='columns')
display(sector_mix.style.format('{:.1%}').background_gradient(cmap='Blues', axis=None))

print("If every column has a similar sector profile, the clusters capture the TYPE of news "
      "(earnings, M&A, analyst notes...) rather than the industry of the company.")

In [ ]:
# Cluster sizes and a compact summary table of the final result
summary = (df_news_unique.groupby('CLUSTER')
           .agg(ARTICLES=('TITLE', 'size'),
                MEAN_SILHOUETTE=('SILHOUETTE', 'mean'),
                TOP_SECTOR=('SECTOR', lambda s: s.value_counts().idxmax()),
                EXAMPLE_TITLE=('TITLE', 'first')))
summary['TOP_TERMS'] = [', '.join(cluster_terms[c][:6]) for c in summary.index]
display(summary[['ARTICLES', 'MEAN_SILHOUETTE', 'TOP_SECTOR', 'TOP_TERMS', 'EXAMPLE_TITLE']]
        .style.format({'MEAN_SILHOUETTE': '{:.3f}'}))

In [ ]:
# The final deliverable: the news DataFrame with embeddings and cluster labels
display(df_news_unique[['TICKER', 'COMPANY', 'SECTOR', 'TITLE', 'PUBLICATION_DATE',
                        'PROVIDER', 'CLUSTER', 'CLUSTER_BEST_K', 'SILHOUETTE']].head(20))

df_news_unique.to_csv('sp500_news_clustered.csv', index=False)
print(f"\nSaved {len(df_news_unique)} clustered articles to sp500_news_clustered.csv")

---
## 📊 Reflection data pack

Every figure the reflection answers should cite, printed in one place. Copy these numbers into the
written answers below so the discussion is anchored in *this* run — the news corpus changes every day,
so a re-run will legitimately produce different clusters, and generic answers score lower than ones
that cite their own evidence.

In [ ]:
print("=" * 78)
print(f"CORPUS      : {len(df_news)} cleaned articles from {df_news['TICKER'].nunique()} tickers "
      f"({df_news['TICKER'].nunique() / len(yahoo_symbols):.1%} of the index)")
print(f"DATE RANGE  : {df_news['PUBLICATION_DATE'].min()} -> {df_news['PUBLICATION_DATE'].max()}")
print(f"CLUSTERED   : {len(df_news_unique)} rows (one per ticker), "
      f"{df_news_unique['TITLE'].str.lower().nunique()} distinct stories")
print(f"EMBEDDINGS  : {X.shape[1]} dimensions ({model.get_sentence_embedding_dimension()}-d model)")
print("-" * 78)
print("CLUSTER-COUNT METRICS")
display(df_silhouette.style.format({
    'SILHOUETTE': '{:.4f}', 'DAVIES_BOULDIN': '{:.4f}',
    'CALINSKI_HARABASZ': '{:.2f}', 'INERTIA': '{:,.0f}'}))
for metric, k in verdicts.items():
    print(f"   {metric:<26} -> k = {k}")
print(f"   inertia falls {df_silhouette['INERTIA'].iloc[0]:,.0f} -> "
      f"{df_silhouette['INERTIA'].iloc[-1]:,.0f} across k = {min(k_values)}..{max(k_values)} "
      f"({1 - df_silhouette['INERTIA'].iloc[-1] / df_silhouette['INERTIA'].iloc[0]:.1%} total drop)")
print("-" * 78)
print(f"PCA         : PC1 {explained[0]:.2%} + PC2 {explained[1]:.2%} = {explained.sum():.2%} "
      f"of the variance shown in the 2D plot")
print(f"FINAL MODEL : k = {K_FINAL}, overall silhouette = {silhouette_score(X, labels_k3):.4f}")
print(f"MISASSIGNED : {(sil_values < 0).sum()} / {len(sil_values)} articles "
      f"({(sil_values < 0).mean():.1%}) have a NEGATIVE silhouette")
print("-" * 78)
print("PER-CLUSTER SUMMARY")
for c in sorted(df_news_unique['CLUSTER'].unique()):
    sub = df_news_unique[df_news_unique['CLUSTER'] == c]
    flag = "  <-- negative mean: not a defensible cluster" if sub['SILHOUETTE'].mean() < 0 else ""
    print(f"   Cluster {c}: n = {len(sub):>3} | mean silhouette = {sub['SILHOUETTE'].mean():+.3f}"
          f" | top sector = {sub['SECTOR'].value_counts().idxmax()}{flag}")
    print(f"      terms: {', '.join(cluster_terms[c][:8])}")
    print(f"      example: {sub.iloc[0]['TITLE'][:88]}")
print("=" * 78)

## Question Section

Take time to reflect on what you've implemented and observed. Answer the following questions in a separate markdown cell or notebook file:

---

### Technical Understanding

#### 1️⃣ How might the choice of embedding model (e.g., MiniLM vs. a larger transformer) affect your clustering results and interpretation?

The embedding model defines the geometry in which K-Means operates, so changing it changes *what
"similar" means* — and therefore the clusters themselves, not merely their quality.

**What MiniLM gives us.** `all-MiniLM-L6-v2` is a 6-layer distilled model producing 384-dimensional
vectors. It is fast (hundreds of sentences per second on CPU), small (~80 MB) and trained on over a
billion sentence pairs, so it captures general semantic similarity very well. For ~500 short
headlines it is more than adequate, and its speed is what makes the whole notebook re-runnable in
minutes.

**What a larger model would change:**

* **Finer semantic resolution.** Larger encoders (`all-mpnet-base-v2`, 768-d; E5/BGE-large, 1024-d)
  separate nuances MiniLM collapses: "beats earnings estimates" vs. "misses earnings estimates" are
  lexically almost identical and differ only in one word, but they are opposite events. A stronger
  model keeps them further apart, which typically produces clusters that are *thematically* rather
  than *lexically* coherent.
* **Domain adaptation matters more than size.** A finance-specific encoder (FinBERT-style, or a model
  fine-tuned on financial text) usually beats a bigger general-purpose model on this corpus, because
  "guidance", "buyback", "downgrade", "dilution" carry technical meanings that general models blur.
  In practice, *domain fit > parameter count*.
* **Higher dimensionality is not free.** Going from 384 to 1024 dimensions worsens the concentration
  of distances (the curse of dimensionality), which tends to *lower* silhouette scores even when the
  clustering is semantically better. Silhouette values are therefore not comparable across models of
  different dimensionality — they must be compared on the same space.
* **Cost and reproducibility.** Bigger models require GPUs for a corpus of any real size, and API
  embeddings (OpenAI, Cohere) add cost, latency and a version-drift risk: a silent model update
  changes the embedding space and breaks comparability with earlier runs.

**Consequence for interpretation.** With MiniLM, clusters tend to organise around *surface topic and
phrasing* (earnings language, analyst-rating language, deal language). With a stronger or
domain-tuned model, clusters shift towards *event type and polarity*, which is far more useful for a
financial application. The right way to decide is not to argue about it but to fix the downstream
task — e.g. "does the cluster label help predict post-news returns?" — and compare models on that
metric, with silhouette used only as a within-model diagnostic.

---

#### 2️⃣ What would be the differences in embeddings if you used only the TITLE, only the SUMMARY, or the combination of both? How could you empirically test this?

**Only the TITLE (what this notebook does).** Titles are short (~10–15 words), dense and written to
convey one idea. The embedding is sharp and low-noise, and clusters tend to be crisp. The costs are
that headlines are stylised ("Shares of X surge as…"), heavily templated by provider, and often
ambiguous without context; and that very short texts give the model little to work with, so lexical
overlap dominates semantics.

**Only the SUMMARY.** Summaries carry more context — the driver of the move, figures, quotes — so the
embedding is richer and better at distinguishing *why* something happened. But they also carry noise:
boilerplate ("Read more on…"), disclaimers, and mentions of several companies at once, which pulls
vectors toward a corpus-average "generic financial text" direction and makes distances more uniform
(lower silhouette). Summaries are also more often missing — the `HAS_SUMMARY` flag computed above
quantifies exactly that in our corpus.

**TITLE + SUMMARY combined.** Concatenation gives the most information but is not automatically the
best: the longer field dominates the average-pooled representation, so the title's signal gets diluted
roughly in proportion to length. Better combination strategies are (a) embedding the two fields
separately and averaging the (normalised) vectors, optionally weighted, which keeps the title's
contribution controllable; or (b) truncating the summary to its first 1–2 sentences, which typically
carry the substance.

**How to test it empirically.** Build the three (or four) embedding variants on the same corpus and
compare them on several axes:

1. **Intrinsic:** silhouette, Davies–Bouldin and Calinski–Harabasz for the same k, plus the stability
   of cluster assignment under bootstrap resampling (Adjusted Rand Index between runs). Note these
   are only comparable when the dimensionality is identical — it is here, since it is the same model.
2. **Agreement between variants:** ARI / Normalised Mutual Information between the partitions
   produced by TITLE-only and SUMMARY-only tells you whether the choice actually matters at all.
3. **Extrinsic — the decisive test:** label a random sample of 150–300 articles with a small taxonomy
   (earnings, M&A, analyst action, legal/regulatory, product, macro) — by hand or with an LLM — and
   measure how well each embedding variant recovers those labels: clustering purity/ARI against the
   human labels, or accuracy of a simple k-NN/logistic classifier trained on the embeddings.
4. **Task-level:** if the downstream goal is financial, evaluate which variant best predicts the sign
   or magnitude of the next-day return, using a strict time-based split.

A clean design fixes the seed, the value of k and the preprocessing across variants so the only thing
that changes is the input text, and repeats each configuration several times to report variability
rather than a single number.

---

#### 3️⃣ In what situations would using a different dimensionality reduction method (e.g., t-SNE, UMAP) be preferable over PCA for visualization of embeddings?

PCA is a **linear** projection that maximises retained variance. Its virtues are that it is
deterministic, fast, invertible, and that its axes are interpretable and preserve *global* structure
and relative distances. Its weakness is visible in this notebook: two components typically capture
only a modest share of the variance of a 384-dimensional embedding space, so a lot of structure is
simply not on the screen, and a linear projection cannot unfold curved (manifold) structure — clusters
that are perfectly separable in 384-d can overlap badly in the 2-d picture.

**Prefer t-SNE when** the goal is to *see whether local neighbourhoods exist at all*. t-SNE preserves
local similarity and produces visually clean, well-separated blobs, which is excellent for exploring
whether the corpus has fine-grained topics. Its caveats matter, though: distances *between* clusters
and cluster sizes are not meaningful, results depend strongly on the `perplexity` parameter and the
seed, and it is slow beyond a few tens of thousands of points. It is a visualisation tool, never an
input to clustering.

**Prefer UMAP when** you want most of t-SNE's local fidelity with better preservation of global
structure, far greater speed on large corpora, and — crucially — the ability to `transform()` new
points with a fitted model, which matters in a production pipeline. UMAP is also frequently used as a
*preprocessing* step (reduce 384-d to 10–50-d, then cluster with HDBSCAN — the standard BERTopic
recipe), because clustering behaves much better in moderate dimension than in 384. Its caveats: it is
stochastic, its `n_neighbors` / `min_dist` parameters visibly shape the picture, and it can manufacture
apparent clusters out of noise.

**In practice for this notebook:** keep PCA as the reference view (it is honest about how much
variance is shown, and it is what makes the "% of variance explained" caption possible), and add UMAP
or t-SNE as a complementary view when the PCA plot looks like one undifferentiated cloud. Note also
that PCA is doing real work beyond plotting here: it is the right tool for *denoising* before
clustering, whereas t-SNE/UMAP coordinates should not be fed into K-Means, because the distances they
produce are distorted by construction.

---

### Data Analysis and Interpretation

#### 4️⃣ Based on your cluster analysis, identify at least two potential challenges you faced in interpreting the clusters and propose strategies to address them.

**Challenge 1 — The clusters overlap, and the silhouette scores are low.** Short-text embeddings of a
homogeneous corpus (all of it is financial news about large caps) do not form well-separated blobs:
silhouette values in the 0.03–0.10 range are typical here, and a non-trivial share of articles get a
*negative* silhouette, meaning they sit closer to another cluster than to their own — the
`NEGATIVE_SILHOUETTE` column above measures exactly this. The consequence is that any label given to
a cluster describes a *tendency*, not a partition: the boundary cases genuinely belong to both groups.
*Strategies:* (a) reduce dimensionality (PCA to 30–50 components) before clustering, to attenuate the
distance-concentration effect; (b) L2-normalise the embeddings so K-Means effectively optimises cosine
similarity, which is the metric the model was trained with; (c) switch to a method that does not force
every point into a cluster — HDBSCAN labels ambiguous articles as noise; or (d) accept soft
assignment (Gaussian Mixture) and report membership probabilities instead of hard labels.

**Challenge 2 — Clusters driven by phrasing rather than by meaning.** MiniLM is sensitive to surface
form, and financial headlines are highly templated. Clusters therefore often align with *stylistic
families* — "Company X Q3 2025 Earnings Call Transcript", "3 Reasons to Buy X Stock", "Analyst
upgrades X" — which are provider formats, not economic events. This is why the cross-tab against GICS
sectors above is informative: if the clusters had matched sectors, they would have been capturing
*who* the news is about rather than *what* happened.
*Strategies:* strip boilerplate patterns and provider signatures before embedding; remove the company
name and ticker from the title so the model cannot cluster on the entity; use a domain-adapted model;
or reframe the task as classification into a predefined event taxonomy, which is what most financial
NLP systems ultimately do.

**Challenge 3 — Naming a cluster is subjective.** Reading ten headlines and declaring "this is the
M&A cluster" is confirmation bias waiting to happen — a different analyst reading a different sample
of the same cluster will name it differently.
*Strategies:* base the label on evidence rather than impression — TF-IDF characteristic terms (as
above), plus the headlines *closest to the centroid* rather than a random sample; have two people label
independently and measure agreement; or generate candidate labels with an LLM over a representative
sample and validate them on a held-out sample of the same cluster.

**Challenge 4 — Instability.** K-Means results depend on initialisation and on k, and the corpus itself
changes every day (the `.news` endpoint returns a rolling window). A theme that looks solid today may
not reappear tomorrow.
*Strategies:* fix `random_state` and use `n_init >= 10` (done here); assess stability by re-clustering
bootstrap samples and measuring the Adjusted Rand Index across runs; and treat any theme that is not
stable across resamples as an artefact rather than a finding.

---

#### 5️⃣ Did you observe any outliers in your 2D visualization? How would you identify and handle these outliers in a production pipeline?

**What to look for in the plot above.** The typical picture is one dense central mass — the
"generic financial headline" region, where most articles about earnings and analyst notes live — with a
thinner periphery of isolated points. Those isolated points are usually one of four things:
(i) non-English or mixed-language headlines; (ii) headlines that are not really news (podcast
episodes, "Live updates", index-rebalancing notices); (iii) genuinely distinctive events (a merger, a
lawsuit, an FDA approval) whose vocabulary is unlike anything else in the corpus; or (iv) very short
or truncated titles carrying almost no signal. Only category (iii) is *interesting*; the rest is
contamination. Two caveats when reading the picture: PCA shows only a fraction of the total variance,
so a point can look extreme in 2-d without being an outlier in 384-d, and vice versa — the honest
detection has to be done in the original space.

**How to identify them systematically (in the full embedding space, not the projection):**

* **Distance-based:** distance to the nearest centroid, or to the k-th nearest neighbour (k-NN
  distance), flagging the upper percentile.
* **Density-based:** Local Outlier Factor, Isolation Forest, or HDBSCAN's explicit noise label — the
  cleanest option, because the model is designed to leave points unassigned.
* **Silhouette-based:** the `SILHOUETTE` column computed above; strongly negative values mark articles
  that do not belong to their assigned cluster.
* **Rule-based pre-filters, which catch most of the junk far more cheaply:** language detection,
  minimum length, deduplication (near-duplicates via cosine similarity > 0.95 — syndicated stories
  appear many times), and a blocklist of known non-news title patterns.

**How to handle them in production.** The key point is that *outliers are not automatically errors*,
so the pipeline should route rather than delete:

1. **Filter at ingestion** what is provably junk (wrong language, too short, duplicate, known
   boilerplate) — cheap and uncontroversial.
2. **Quarantine, don't drop, the rest.** Assign them to a "noise / unclustered" bucket and keep them in
   the database with their score, so they can be reviewed and so nothing is silently lost. In finance
   the rare, unusual article is often the one that matters most — an outlier detector is also a
   *novelty* detector.
3. **Cluster robustly:** fit the centroids on the clean core, then assign the remaining points, or use
   methods with a built-in noise class instead of K-Means, whose centroids are sensitive to extreme
   points.
4. **Monitor.** Track the share of outliers over time: a sudden rise usually means an upstream change
   (the provider altered its format, a new syndication source appeared, the API schema changed) rather
   than a change in the market. Alert on drift instead of discovering it months later.
5. **Escalate the interesting ones.** Route high-novelty items to a human analyst or to a
   more expensive model — that is where the alpha of a news pipeline usually is.

---

#### 6️⃣ If you could assign a 'label' or 'theme' to each cluster you obtained, what would they be? How confident are you in these assignments, and what could you do to validate them systematically?

**The labels.** The evidence for naming the clusters is produced above by the *characteristic terms*
(TF-IDF centroids) and the *most representative headlines* (highest silhouette within each cluster) —
those two outputs are what a label must be based on. With `k = 3` on a corpus of one recent headline
per S&P 500 company, the recurring themes are usually along these lines:

* **A "corporate results and guidance" cluster** — earnings beats/misses, revenue, guidance, earnings
  call transcripts. It is normally the largest, because earnings coverage dominates financial news.
* **An "analyst / market action" cluster** — upgrades, downgrades, price targets, "stock moves", "why
  shares jumped", best/worst performers. Highly templated language, which is why it separates well.
* **A "corporate events and strategy" cluster** — M&A, buybacks, dividends, leadership changes,
  litigation, regulation, product launches. It is usually the most heterogeneous and the one with the
  lowest mean silhouette.

> 📋 **After running the notebook, replace the three bullets above with the labels your own run
> produces** — the exact themes depend on the day the news is pulled, and the characteristic terms
> printed by the analysis cells are the evidence to cite.

**How confident am I?** *Moderately, and deliberately so.* Three reasons for caution: the silhouette
scores are low, so the boundaries between these themes are genuinely fuzzy and a meaningful share of
articles could be reassigned without much loss; `k = 3` is imposed by the exercise rather than by the
data (compare with `CLUSTER_BEST_K` — if the silhouette-optimal k differs, the three-way split is a
convention, not a discovery); and reading the top-10 headlines of a cluster is a small, non-random
sample that invites confirmation bias. The labels are a useful summary of a tendency, not a
classification I would put into production untested.

**Systematic validation:**

1. **Manual gold standard.** Label a random sample of 200–300 articles against a predefined taxonomy,
   with two independent annotators; measure inter-annotator agreement (Cohen's κ) to establish how
   well-defined the taxonomy itself is, then measure cluster purity, ARI and NMI against those labels.
2. **LLM-assisted labelling.** Use an LLM to classify a larger sample into the same taxonomy — cheaper
   than human labelling and consistent — and validate the LLM against the human subset first.
3. **Held-out confirmation.** Name the clusters from one sample and test the names on a *different*
   sample of the same clusters: if the label does not predict the content of unseen members, it is a
   narrative, not a finding.
4. **Stability testing.** Bootstrap-resample the corpus, re-cluster, and check with ARI whether the
   same themes reappear; run across several days of news and check the themes persist over time.
5. **Sanity checks against known structure.** The sector cross-tab above is one: if clusters simply
   tracked GICS sectors, the "themes" would be an artefact of company identity rather than of news
   content.
6. **Extrinsic usefulness.** The strongest validation is that the label carries information: do
   articles in the "analyst action" cluster show different average next-day returns or volumes than
   those in the "results" cluster? A theme that changes nothing downstream is decoration.

---

### Critical Thinking

#### 7️⃣ If news sentiment was incorporated into the analysis, how might this influence the clustering structure and interpretation of the clusters in a financial analysis context?

**Sentiment is largely orthogonal to topic, and that is precisely what makes it valuable.** Semantic
embeddings place "Company X beats earnings estimates" and "Company X misses earnings estimates" very
close together — same topic, same vocabulary, opposite economic meaning. Purely semantic clusters are
therefore *event-type* clusters that mix good and bad news, which is a real limitation for a financial
application, where direction is often what matters most.

**What changes if sentiment is added:**

* **Adding a sentiment dimension to the feature vector** (or clustering within each topic cluster) turns
  a 3-cluster topical partition into something closer to a *topic × polarity* grid: positive earnings
  vs. negative earnings, upgrades vs. downgrades, favourable vs. adverse legal outcomes. Cluster count
  rises roughly multiplicatively, and the clusters become directly actionable.
* **The interpretation changes from "what is being discussed" to "what is being priced".** "Cluster 1 =
  results" becomes "Cluster 1a = beats with raised guidance" and "Cluster 1b = misses with cut
  guidance" — two groups with opposite expected return signatures.
* **Weighting matters.** A single sentiment scalar appended to 384 dimensions will be ignored by
  Euclidean K-Means unless it is scaled up substantially; the honest approaches are to standardise and
  weight it explicitly, or to cluster hierarchically (topic first, then split by sentiment), which also
  keeps the result interpretable.
* **Aggregate views become possible.** Sentiment per cluster, per sector and over time turns the
  analysis into a monitoring tool: "negative sentiment in the regulatory cluster is concentrating in
  Financials this week" is an insight the topical clustering alone cannot produce.

**Financial caveats that must accompany any such extension.** General-purpose sentiment models are
unreliable on financial text — "the company reported a loss narrower than expected" is *good* news
with negative surface wording — so a finance-tuned model (FinBERT and its successors) is required.
Sentiment is also endogenous to price: much coverage is written *after* the move ("shares surge on…"),
so a sentiment feature can be a lagging proxy for returns rather than a predictor, and only
point-in-time data with a strict timestamp discipline can settle that. And what moves prices is not
sentiment in absolute terms but sentiment *relative to expectations* — a strongly positive article
about a company everyone already expected to do well carries little information.

---

#### 8️⃣ Discuss the limitations of using k-means clustering for news embeddings. What alternative clustering methods could address these limitations, and under what conditions would you prefer them?

**Limitations of K-Means here:**

1. **k must be chosen in advance**, and the number of genuine themes in a news corpus is unknown and
   changes daily. The silhouette curve computed above is a heuristic, and on short-text embeddings it
   is often flat — which is itself evidence that no single k is clearly right.
2. **Every point is forced into a cluster.** There is no notion of noise, so irrelevant or unique
   articles are absorbed into whatever centroid is closest, blurring cluster identity and dragging
   centroids around.
3. **Spherical, equal-variance, convex clusters are assumed.** Real topics are neither balanced nor
   spherical: "earnings" is huge and diffuse, "FDA approvals" is small and tight. K-Means will split
   the big cluster and merge the small ones, because it optimises within-cluster variance.
4. **Euclidean distance vs. cosine semantics.** Sentence embeddings are trained for *cosine*
   similarity; K-Means minimises squared Euclidean distance. The two coincide only if the vectors are
   L2-normalised — normalising first (spherical K-Means) is the standard fix and is worth doing here.
5. **Curse of dimensionality.** In 384 dimensions distances concentrate, so all points look roughly
   equidistant, silhouettes collapse toward zero, and the partition is less meaningful than the
   numbers suggest.
6. **Sensitivity to initialisation and to outliers**, since centroids are means — mitigated but not
   eliminated by `k-means++` and `n_init=10`.

**Alternatives and when to prefer them:**

* **HDBSCAN** (density-based, hierarchical): finds clusters of varying density and shape, chooses the
  number of clusters itself, and — decisively — labels ambiguous articles as *noise*. Preferred when
  the corpus is messy and you would rather have three clean themes plus a noise bucket than five
  forced ones. Standard practice is to reduce to 5–50 dimensions with UMAP first, because density
  estimation degrades badly in high dimension.
* **Agglomerative / hierarchical clustering** with cosine distance and average linkage: gives a
  dendrogram, so the granularity can be chosen *after* seeing the structure, and nested themes
  ("results" → "results with guidance cut") become visible. Preferred for exploratory analysis and for
  corpora of a few thousand documents, where its O(n²) cost is acceptable.
* **Gaussian Mixture Models:** soft assignment with membership probabilities and elliptical clusters —
  a better fit conceptually, since an article about a merger *and* an earnings beat genuinely belongs
  to two themes. Preferred when downstream consumers can use probabilities rather than hard labels.
* **Spectral clustering:** handles non-convex geometry through a similarity graph; preferred for small
  corpora with clearly non-spherical structure, but does not scale.
* **BERTopic** (UMAP + HDBSCAN + class-based TF-IDF): the pragmatic production choice for exactly this
  task, since it delivers clusters *and* interpretable topic descriptors, handles noise, and supports
  incremental/online updates as news arrives.
* **Topic models (LDA)** remain a reasonable baseline on longer texts, though they underperform
  embedding-based methods on short headlines.
* **Supervised classification into a fixed taxonomy** — the honest answer when the business need is
  stable ("is this an M&A story?"). Clustering is for discovery; once the categories are known, a
  classifier is more accurate, more stable and easier to monitor.

**When K-Means is still the right call:** it is fast, deterministic under a fixed seed, trivial to
explain, and gives a usable first map of a corpus. For an exercise of ~500 headlines it is a perfectly
defensible baseline — the point is to know its assumptions and to report results with the appropriate
level of confidence.

---

#### 9️⃣ How could the approach in this notebook be extended to analyze the potential impact of news clusters on stock price movements over time? Sketch a high-level pipeline you would implement to test this.

The question becomes an **event study**, structurally identical to the one applied to Golden Crosses
in Section B — but with the news cluster as the event.

**High-level pipeline**

1. **Historical, point-in-time news collection.** The `.news` endpoint only returns a short rolling
   window, so a real study needs a historical archive with reliable timestamps (RavenPack, Refinitiv,
   GDELT, or a self-built store accumulated daily). *Point-in-time* is non-negotiable: the timestamp
   must be the moment the article became public, not a later revision.
2. **Deduplicate and normalise.** Syndicated stories appear dozens of times; without near-duplicate
   removal (cosine > 0.95), a single event is counted many times and the statistics are meaningless.
3. **Embed and assign clusters.** Fit the embedding + clustering model on a *training period only*,
   then `transform`/`predict` for later periods. Refitting on the whole history leaks future
   information into past labels — this is the single most common way such a study becomes invalid.
   Add sentiment as a separate feature at this stage.
4. **Align news with market data.** For each article, attach the ticker's price series and define the
   event time correctly: an article published after the close maps to the *next* session's open. Compute
   forward returns over several horizons (1, 5, 10, 21 sessions) and, critically, **abnormal** returns —
   raw returns are dominated by market and sector moves. Use a market model (CAPM/Fama-French residuals)
   or simply subtract sector-ETF returns.
5. **Aggregate by cluster.** For each cluster, compute mean/median cumulative abnormal return (CAR) over
   an event window (e.g. −5 to +21 days), the hit rate, abnormal volume and abnormal volatility, and
   plot average CAR paths per cluster. The pre-event window is a built-in diagnostic: if CAR is already
   drifting *before* publication, the news is following the price, not leading it.
6. **Test significance properly.** Cross-sectional t-tests are the minimum; a bootstrap over event
   dates is better, because events cluster in time (earnings season) and are therefore not independent.
   Control for firm size, sector, volatility and calendar effects, and correct for multiple testing
   across clusters and horizons.
7. **Validate out-of-sample and across regimes.** Split by time (train ≤ year *T*, test > *T*), and check
   the effect survives in a different market regime. Anything that only works in-sample is noise.
8. **Turn it into a strategy only if step 7 passes.** Simulate with realistic entry (next open), costs
   (spread + commission + slippage) and capacity limits; evaluate Sharpe, drawdown and turnover
   against buy-and-hold. Combine with the technical signals from Section B — the natural hypothesis is
   that a Golden Cross confirmed by a positive news cluster is more reliable than either alone.
9. **Productionise:** daily incremental ingestion, drift monitoring on cluster proportions and on
   embedding distributions, model versioning (so that a model upgrade does not silently change the
   labels), and periodic re-validation.

**The three traps that invalidate most such studies:** look-ahead bias (fitting the clusters on the
full history, or using revised timestamps); the fact that news is largely priced within *minutes* for
liquid large caps, so daily-horizon "predictive power" is often just momentum in disguise; and
survivorship bias in the universe. Expect small effects — the realistic hypothesis is a modest,
short-lived drift after certain event types, not a tradable signal on its own.

---

#### 🔟 Imagine your clustering shows clear groups of news, but your downstream task (e.g., prediction of stock movement) does not improve. What might explain this disconnect between clear clusters and predictive utility?

This is the normal outcome, and it is worth stating plainly: **cluster separability and predictive
power are different properties, optimised by different objectives.** K-Means maximises geometric
compactness in an embedding space trained for *semantic similarity*; nothing in that objective
mentions returns. Clear clusters mean the text is structured, not that the structure is priced.

The main explanations:

1. **The clusters capture style, not information.** Well-separated groups often correspond to article
   *formats* — "Q3 Earnings Call Transcript", "3 Reasons to Buy X" — which are properties of the
   publisher, not of the company's prospects. Perfectly separable, perfectly uninformative.
2. **Market efficiency.** For S&P 500 constituents, public news is incorporated into prices within
   seconds to minutes. By the time an article is retrievable from a free API and assigned to a cluster,
   the information is in the price. Any residual signal lives at a horizon the pipeline cannot trade.
3. **The signal is direction, and the clustering is topic.** "Earnings news" contains both beats and
   misses; the cluster label sums to roughly zero expected return by construction. Without polarity
   *relative to expectations*, the feature cannot predict the sign of the move. The fix is sentiment or
   surprise, not more clusters.
4. **Granularity mismatch.** Three clusters over a heterogeneous corpus is an extremely coarse feature —
   near-zero information content for a regression. What matters may be a rare sub-theme (bankruptcy
   filings, FDA rejections) that is invisible inside a large generic cluster.
5. **Reverse causality.** Much financial coverage is written *because* the price moved. Such features
   correlate with *past* returns and appear useful in-sample while having no forward-looking value.
6. **Signal-to-noise and sample size.** Daily equity returns are ~95% noise; detecting a small effect
   needs thousands of observations. One headline per company on one day is nowhere near enough
   statistical power — a true effect would not be detectable even if present.
7. **Aggregation and timing loss.** Reducing an article to a cluster ID discards almost all of its
   content, and the daily bucketing discards the intraday timing that carries the reaction.
8. **Leakage in the *other* direction.** Sometimes the clusters look clear only because the model was
   fit on the whole corpus including the test period; the apparent structure does not generalise, so
   the predictive test fails while the visualisation looks great.

**What to do about it.** Test the pieces separately: first check whether *any* text-derived feature
predicts returns (a supervised model trained end-to-end on the embeddings, not on cluster IDs — if the
embedding contains signal, the classifier will find it, and the loss of information from clustering is
the likely culprit); measure abnormal returns rather than raw ones; move to intraday horizons; add
surprise/sentiment features; and be prepared to accept the null result. A clustering that organises a
news feed for human analysts is genuinely useful even if it predicts nothing — unsupervised structure
is an *exploration* tool, and the honest conclusion is often that the value is in monitoring and
triage rather than in alpha.